# Vector Databases — AI Engineering Interview Prep

FAISS and ChromaDB with sentence-transformers for semantic search.

**Install**: `pip install faiss-cpu chromadb sentence-transformers`

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

model = SentenceTransformer('all-MiniLM-L6-v2')
DIM = model.get_sentence_embedding_dimension()
print(f"Embedding model: all-MiniLM-L6-v2, dim={DIM}")

# Corpus of ML/AI documents
corpus = [
    "Gradient descent is an optimization algorithm that minimizes a loss function by updating parameters in the direction of the negative gradient.",
    "Transformers use self-attention to relate different positions of a sequence when computing representations.",
    "Overfitting occurs when a model learns the training data too well, including noise, leading to poor generalization.",
    "Convolutional neural networks are designed for grid-structured data like images, using local receptive fields.",
    "Batch normalization normalizes activations across a mini-batch, stabilizing and accelerating training.",
    "Dropout randomly deactivates neurons during training as a regularization technique to prevent overfitting.",
    "BERT is a bidirectional transformer pretrained using masked language modeling and next sentence prediction.",
    "Adam optimizer combines the benefits of AdaGrad and RMSProp, adapting learning rates per parameter.",
    "Transfer learning reuses a model trained on one task as the starting point for a different task.",
    "Reinforcement learning trains agents through interaction with an environment using reward signals.",
    "Support vector machines find the hyperplane that maximizes the margin between classes.",
    "Random forests aggregate predictions from many decision trees trained on bootstrap samples.",
    "K-means clustering partitions data into K groups by minimizing within-cluster sum of squared distances.",
    "Word2Vec learns word embeddings by predicting context words (skip-gram) or target words (CBOW).",
    "Attention mechanisms allow models to focus on relevant parts of the input when generating each output token.",
]

corpus_embeddings = model.encode(corpus, normalize_embeddings=True)
print(f"Corpus: {len(corpus)} documents, embeddings shape: {corpus_embeddings.shape}")

## 1. FAISS — Flat Index (Exact Search)

In [ ]:
import faiss

# IndexFlatIP = inner product (== cosine sim for normalized vectors)
index_flat = faiss.IndexFlatIP(DIM)
index_flat.add(corpus_embeddings.astype(np.float32))
print(f"FAISS Flat index: {index_flat.ntotal} vectors")

def faiss_search(query, index, k=3):
    q_emb = model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    return [(corpus[i], float(scores[0][j])) for j, i in enumerate(indices[0])]

queries = [
    "how does attention work in neural networks?",
    "techniques to prevent overfitting",
    "optimization algorithms for deep learning",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    for doc, score in faiss_search(q, index_flat):
        print(f"  [{score:.4f}] {doc[:80]}...")

## 2. FAISS — HNSW Index (Approximate, Fast)

In [ ]:
# HNSW: Hierarchical Navigable Small World — approximate nearest neighbor
M = 16  # number of connections per layer
index_hnsw = faiss.IndexHNSWFlat(DIM, M)
index_hnsw.hnsw.efConstruction = 200  # build time accuracy/speed tradeoff
index_hnsw.hnsw.efSearch = 50        # search time accuracy/speed tradeoff
index_hnsw.add(corpus_embeddings.astype(np.float32))

# Scale test: generate 10K synthetic vectors
big_vectors = np.random.randn(10000, DIM).astype(np.float32)
faiss.normalize_L2(big_vectors)

flat_big = faiss.IndexFlatIP(DIM)
hnsw_big = faiss.IndexHNSWFlat(DIM, M)
flat_big.add(big_vectors)
hnsw_big.add(big_vectors)

query_vec = np.random.randn(1, DIM).astype(np.float32)
faiss.normalize_L2(query_vec)

t0 = time.perf_counter()
for _ in range(100): flat_big.search(query_vec, 10)
flat_time = (time.perf_counter() - t0) / 100 * 1000

t0 = time.perf_counter()
for _ in range(100): hnsw_big.search(query_vec, 10)
hnsw_time = (time.perf_counter() - t0) / 100 * 1000

print(f"10K vectors, 100 queries averaged:")
print(f"  Flat (exact): {flat_time:.3f} ms")
print(f"  HNSW (approx): {hnsw_time:.3f} ms")
print(f"  Speedup: {flat_time/hnsw_time:.1f}x")

## 3. ChromaDB — Persistent Vector Store

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# In-memory client (use chromadb.PersistentClient(path='./chroma_db') for disk)
chroma_client = chromadb.Client()

# Use sentence-transformers embedding function
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

collection = chroma_client.create_collection(
    name="ml_docs",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)

# Add documents with metadata
collection.add(
    documents=corpus,
    ids=[f"doc_{i}" for i in range(len(corpus))],
    metadatas=[{"category": "optimization" if any(w in d for w in ["gradient", "Adam", "optimizer"]) else
                            "architecture" if any(w in d for w in ["transformer", "CNN", "BERT", "attention"]) else
                            "regularization" if any(w in d for w in ["overfitting", "dropout", "normalization"]) else
                            "general"} for d in corpus]
)

print(f"ChromaDB collection: {collection.count()} documents")

# Basic query
results = collection.query(query_texts=["how to prevent overfitting"], n_results=3)
print("\nQuery: 'how to prevent overfitting'")
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    print(f"  [{1-dist:.4f}] [{meta['category']}] {doc[:70]}...")

# Filter by metadata
filtered = collection.query(
    query_texts=["neural network training"],
    n_results=3,
    where={"category": "optimization"}
)
print("\nFiltered (optimization only):")
for doc in filtered['documents'][0]:
    print(f"  {doc[:70]}...")

## 4. Index Types Comparison

In [ ]:
comparison = {
    "FAISS Flat": {
        "type": "Exact", "speed": "O(N)", "memory": "Low",
        "accuracy": "100%", "best_for": "<1M vectors, high accuracy needed"
    },
    "FAISS HNSW": {
        "type": "Approximate", "speed": "O(log N)", "memory": "High (graph)",
        "accuracy": "95-99%", "best_for": "<100M vectors, fast search"
    },
    "FAISS IVF": {
        "type": "Approximate", "speed": "O(N/nlist)", "memory": "Low",
        "accuracy": "90-98%", "best_for": "Large datasets, CPU"
    },
    "ChromaDB": {
        "type": "Managed", "speed": "HNSW-based", "memory": "Medium",
        "accuracy": "High", "best_for": "Prototyping, metadata filtering"
    },
    "Pinecone/Weaviate": {
        "type": "Managed Cloud", "speed": "Fast", "memory": "Unlimited",
        "accuracy": "High", "best_for": "Production, >100M vectors"
    },
}

print(f"{'Index':<22} {'Type':<12} {'Speed':<12} {'Memory':<10} {'Best For'}")
print("-" * 90)
for name, info in comparison.items():
    print(f"{name:<22} {info['type']:<12} {info['speed']:<12} {info['memory']:<10} {info['best_for']}")

## Interview Tips

- **Flat vs ANN**: Flat is exact but O(N). Use ANN (HNSW/IVF) for >100K vectors.
- **Normalize embeddings**: required for cosine similarity to work as inner product.
- **HNSW tradeoffs**: `efSearch` controls recall vs speed at query time. `efConstruction` controls build quality.
- **Chunking strategy**: how you split documents matters more than the vector DB choice.
- **Metadata filtering**: pre-filter with metadata before ANN search to reduce candidate set.
- **Production DBs**: Pinecone, Weaviate, Qdrant — all use HNSW with persistence, filtering, and scaling.